# Übung 03 – Kundensegmentierung mit Clustering: Wholesale Customers

## Lernziel

Ich durchlaufe einen vollständigen, nachvollziehbaren Clustering-Workflow: Daten verstehen, für Distanzen geeignet vorbereiten, plausible Clusterzahlen vergleichen, Segmente inhaltlich profilieren und die Grenzen der gefundenen Struktur offen benennen.

## Datensatz und Quelle

| Angabe | Quelle |
|---|---|
| Offizielle Dokumentation | https://archive.ics.uci.edu/dataset/292/wholesale+customers |
| Direkter Download | https://archive.ics.uci.edu/static/public/292/wholesale+customers.zip |
| DOI | https://doi.org/10.24432/C5030X |
| Zitierform | Cardoso, M. (2013). *Wholesale customers* [Dataset]. UCI Machine Learning Repository. |
| Lizenz laut UCI | CC BY 4.0 |

## Fallfrage

**Welche Ausgabenmuster lassen sich im Kundenstamm eines Großhändlers erkennen und wie lassen sich die resultierenden Segmente fachlich beschreiben?**

`Channel` und `Region` werden bewusst nicht zum Clustering verwendet. Sie dienen erst im Anschluss zur Plausibilisierung. Würde ich sie als Eingabe verwenden, würde ich die gesuchte Struktur teilweise bereits vorgeben.

Als methodische Grundlage für die Silhouette siehe Rousseeuw, P. J. (1987). *Journal of Computational and Applied Mathematics, 20*, 53–65. https://doi.org/10.1016/0377-0427(87)90125-7.

In [ ]:
# Ich halte alle Imports am Anfang zusammen. Dadurch ist transparent,
# welche Bibliotheken mein Notebook benötigt und ich kann Fehler schneller eingrenzen.
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile
import shutil
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# Die feste Zufallszahl macht zufällige Aufteilungen und Modellresultate reproduzierbar.
SEED = 42
np.random.seed(SEED)

# Diese Darstellung ist für die Analyse in Colab gut lesbar.
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')

DATA_DIR = Path('daten')
DATA_DIR.mkdir(exist_ok=True)


def download_and_extract_zip(url: str, label: str) -> Path:
    """Lädt ein offizielles ZIP-Archiv nur bei Bedarf herunter und entpackt es.

    Die Funktion ist absichtlich im Notebook sichtbar: Studierende sollen erkennen,
    dass die Datenquelle nicht manuell und nicht über einen lokalen Pfad bereitgestellt wird.
    """
    zip_path = DATA_DIR / f'{label}.zip'
    extract_dir = DATA_DIR / label

    if not zip_path.exists():
        print(f'Lade Daten von: {url}')
        response = requests.get(url, timeout=120)
        response.raise_for_status()
        zip_path.write_bytes(response.content)
    else:
        print(f'Verwende vorhandenes Archiv: {zip_path}')

    if not extract_dir.exists():
        extract_dir.mkdir(parents=True)
        with ZipFile(zip_path) as archive:
            archive.extractall(extract_dir)

    return extract_dir

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## 1. Daten laden und Struktur verstehen

Die sechs Ausgabenmerkmale sind Beträge und unterscheiden sich stark in Größenordnung und Schiefe. Bevor ich eine Distanz berechne, prüfe ich daher Verteilungen und entscheide anschließend über Transformation und Skalierung.

In [ ]:
SOURCE_URL = 'https://archive.ics.uci.edu/static/public/292/wholesale+customers.zip'
extract_dir = download_and_extract_zip(SOURCE_URL, 'wholesale_customers')
csv_path = extract_dir / 'Wholesale customers data.csv'
assert csv_path.exists(), f'Die erwartete Datei fehlt: {csv_path}'

df = pd.read_csv(csv_path)
print(f'Datensatzform: {df.shape[0]:,} Zeilen und {df.shape[1]} Spalten')
df.head()

In [ ]:
spend_features = ['Fresh', 'Milk', 'Grocery', 'Frozen', 'Detergents_Paper', 'Delicassen']
context_features = ['Channel', 'Region']

print('Fehlende Werte je Spalte:')
display(df.isna().sum().to_frame('fehlende Werte'))
print('\\nDeskriptive Kennzahlen der Ausgabenmerkmale:')
display(df[spend_features].describe().T.round(1))

## 2. Verteilungen und Transformation

Die Verteilungen sind rechtsschief: einzelne Kund:innen geben in manchen Kategorien deutlich mehr aus als die Mehrheit. Ich verwende `log1p`, also `log(1 + x)`. Das funktioniert auch bei einem hypothetischen Wert von 0 und reduziert den dominierenden Einfluss extrem großer Beträge. Die Transformation ist eine fachliche Entscheidung; ich dokumentiere sie deshalb sichtbar.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for feature, ax in zip(spend_features, axes.ravel()):
    sns.histplot(df[feature], bins=30, kde=True, color='#3b7ddd', ax=ax)
    ax.set_title(feature)
plt.suptitle('Ausgabenverteilungen vor der Log-Transformation', y=1.02)
plt.tight_layout()
plt.show()

X_raw = df[spend_features].copy()
X_log = np.log1p(X_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(X_log.corr(), cmap='vlag', center=0, square=True, ax=ax)
ax.set_title('Korrelationen der log-transformierten Ausgabenmerkmale')
plt.show()

## 3. Wie viele Cluster sind plausibel?

K-Means benötigt die Clusterzahl `k` vorab. Ich vergleiche deshalb mehrere Werte. Die mittlere Silhouettenbreite verbindet die Frage „Wie eng liegen Fälle innerhalb ihres Clusters?“ mit der Frage „Wie gut sind Cluster voneinander getrennt?“. Sie ist ein Hilfsmittel, keine automatische Geschäftsentscheidung.

In [ ]:
k_values = range(2, 8)
silhouette_rows = []
models = {}

for k in k_values:
    kmeans = KMeans(n_clusters=k, n_init=30, random_state=SEED)
    labels = kmeans.fit_predict(X_scaled)
    silhouette_rows.append({'k': k, 'silhouette': silhouette_score(X_scaled, labels), 'inertia': kmeans.inertia_})
    models[k] = kmeans

cluster_quality = pd.DataFrame(silhouette_rows)
display(cluster_quality.style.format({'silhouette': '{:.3f}', 'inertia': '{:.1f}'}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=cluster_quality, x='k', y='silhouette', marker='o', ax=axes[0])
axes[0].set(title='Mittlere Silhouettenbreite', xlabel='Anzahl Cluster k', ylabel='Silhouette')
sns.lineplot(data=cluster_quality, x='k', y='inertia', marker='o', ax=axes[1])
axes[1].set(title='Inertia als Zusatzinformation', xlabel='Anzahl Cluster k', ylabel='Inertia')
plt.tight_layout()
plt.show()

## 4. Segmentlösung auswählen und visualisieren

Ich wähle hier die Clusterzahl mit der höchsten Silhouette als **technischen Startpunkt**. Vor einer endgültigen Entscheidung würde ich zusätzlich prüfen, ob die Segmente stabil, verständlich und für den Geschäftskontext nutzbar sind. Für die 2D-Grafik verwende ich PCA nur zur Visualisierung; das eigentliche Clustering wurde auf allen sechs vorbereiteten Merkmalen durchgeführt.

In [ ]:
best_k = int(cluster_quality.loc[cluster_quality['silhouette'].idxmax(), 'k'])
final_kmeans = models[best_k]
df['cluster'] = final_kmeans.labels_

print(f'Ausgewählte Clusterzahl als Startpunkt: k = {best_k}')
print('Clustergrößen:')
display(df['cluster'].value_counts().sort_index().to_frame('Anzahl Kund:innen'))

pca = PCA(n_components=2, random_state=SEED)
coordinates = pca.fit_transform(X_scaled)
plot_df = pd.DataFrame({'PC1': coordinates[:, 0], 'PC2': coordinates[:, 1], 'cluster': df['cluster'].astype(str)})

fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=plot_df, x='PC1', y='PC2', hue='cluster', palette='tab10', alpha=0.75, ax=ax)
ax.set(title='Cluster in einer zweidimensionalen PCA-Darstellung')
plt.show()

## 5. Segmente profilieren und nachträglich plausibilisieren

Ein Cluster wird erst durch ein Profil verständlich. Ich vergleiche deshalb Median-Ausgaben in den ursprünglichen Geldeinheiten. Danach nutze ich `Channel` ausschließlich als **nachgelagerte Plausibilisierung**, nicht als Input. Ein Zusammenhang beweist nicht, dass der Kanal die Segmente verursacht.

In [ ]:
cluster_profile = df.groupby('cluster')[spend_features].median().round(0)
display(cluster_profile.style.background_gradient(cmap='Blues', axis=None))

profile_long = cluster_profile.reset_index().melt(id_vars='cluster', var_name='Kategorie', value_name='Median-Ausgabe')
fig, ax = plt.subplots(figsize=(13, 5))
sns.barplot(data=profile_long, x='Kategorie', y='Median-Ausgabe', hue='cluster', ax=ax)
ax.set(title='Segmentprofile: Median-Ausgaben nach Produktkategorie', xlabel='Produktkategorie', ylabel='Median-Ausgabe')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()

channel_check = pd.crosstab(df['cluster'], df['Channel'], normalize='index').mul(100).round(1)
print('Nachgelagerte Plausibilisierung: Kanalanteile je Cluster in Prozent')
display(channel_check)

## 6. Transferfragen

1. Warum wären unskalierte Ausgabenbeträge für K-Means problematisch?
2. Warum wird `Channel` nicht als Clustering-Merkmal verwendet?
3. Warum reicht die höchste Silhouettenbreite allein nicht für eine endgültige Geschäftsentscheidung?
4. Formuliere für ein Cluster eine präzise, nicht wertende Segmentbeschreibung.

> **Merksatz:** Clustering entdeckt Strukturen in Daten. Ob diese Struktur stabil, sinnvoll und handlungsrelevant ist, muss anschließend fachlich geprüft werden.